# 05 - Benchmark & Ablation Study (Paper-Quality)

## Part A: Benchmark Aggregation
- Loads results from NB02 (Classical), NB03 (U-Net), NB04 (3DGS)
- Generates paper-ready comparison tables
- Statistical significance tests

## Part B: Ablation Study
Trains 3DGS with 4 variants to validate contributions:
1. **No regularization**: lambda_smooth=0, lambda_edge=0
2. **Smoothness only**: lambda_smooth=0.01, lambda_edge=0
3. **Edge only**: lambda_smooth=0, lambda_edge=0.005
4. **Full (Ours)**: lambda_smooth=0.01, lambda_edge=0.005

Also ablates:
5. **Grid init vs Adaptive init**
6. **With vs without LR decay**

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True
SEED = 42

# Ablation config
ABLATION_SPARSE_RATIO = 3  # R=3 shows differences most clearly

if not FULL_EXPERIMENT:
    ABLATION_CASES = 2       # Number of test cases for ablation
    ABLATION_ITERATIONS = 500
else:
    ABLATION_CASES = 10      # Subset for ablation (time-intensive)
    ABLATION_ITERATIONS = 3000
# ===================================================================

In [ ]:
import os, sys, json, subprocess, time, gc
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from copy import deepcopy

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"
if not (REPO_DIR / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.utils.seed import set_seed
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Part A: Benchmark Aggregation

In [ ]:
# Load all result CSVs from previous notebooks
dfs = []

classical_csv = OUTPUT_ROOT / "classical_baselines" / "summary.csv"
if classical_csv.exists():
    dfs.append(pd.read_csv(classical_csv))
    print(f"Classical: {len(dfs[-1])} entries")

unet_csv = OUTPUT_ROOT / "unet_baseline" / "summary.csv"
if unet_csv.exists():
    dfs.append(pd.read_csv(unet_csv))
    print(f"U-Net: {len(dfs[-1])} entries")

dgs_csv = OUTPUT_ROOT / "3dgs" / "summary.csv"
if dgs_csv.exists():
    dfs.append(pd.read_csv(dgs_csv))
    print(f"3DGS: {len(dfs[-1])} entries")

if not dfs:
    print("No results found. Run notebooks 02-04 first.")
else:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal entries: {len(df_all)}")
    print(f"Methods: {df_all['method'].unique()}")
    print(f"Sparse ratios: {sorted(df_all['sparse_ratio'].unique())}")

In [ ]:
if dfs:
    # ============= PAPER TABLE 1: Main Results =============
    print("\n" + "="*80)
    print("  PAPER TABLE 1: Quantitative Comparison of Interpolation Methods")
    print("="*80)

    # Method order for paper
    method_order = ["nearest", "linear", "cubic", "unet", "3dgs"]
    method_names = {
        "nearest": "Nearest",
        "linear": "Linear",
        "cubic": "Cubic",
        "unet": "U-Net 2D",
        "3dgs": "3DGS (Ours)",
    }

    for R in sorted(df_all["sparse_ratio"].unique()):
        print(f"\n--- R={R} ---")
        sub = df_all[df_all["sparse_ratio"] == R]
        for method in method_order:
            m = sub[sub["method"] == method]
            if len(m) == 0:
                continue
            name = method_names.get(method, method)
            psnr_mean = m["mean_psnr"].mean()
            psnr_std = m["mean_psnr"].std()
            ssim_mean = m["mean_ssim"].mean()
            ssim_std = m["mean_ssim"].std()
            mae_mean = m["mean_mae"].mean() if "mean_mae" in m.columns else float('nan')
            print(f"  {name:<15} | PSNR: {psnr_mean:.2f} +/- {psnr_std:.2f} | "
                  f"SSIM: {ssim_mean:.4f} +/- {ssim_std:.4f} | MAE: {mae_mean:.4f}")

    # Save formatted table
    table_rows = []
    for R in sorted(df_all["sparse_ratio"].unique()):
        sub = df_all[df_all["sparse_ratio"] == R]
        for method in method_order:
            m = sub[sub["method"] == method]
            if len(m) == 0:
                continue
            table_rows.append({
                "R": R,
                "Method": method_names.get(method, method),
                "PSNR": f"{m['mean_psnr'].mean():.2f} ({m['mean_psnr'].std():.2f})",
                "SSIM": f"{m['mean_ssim'].mean():.4f} ({m['mean_ssim'].std():.4f})",
                "MAE": f"{m['mean_mae'].mean():.4f}" if "mean_mae" in m.columns else "-",
            })
    df_table = pd.DataFrame(table_rows)
    df_table.to_csv(OUTPUT_ROOT / "paper_table1.csv", index=False)
    print("\nSaved paper_table1.csv")

In [ ]:
if dfs:
    # ============= RUNTIME COMPARISON TABLE =============
    print("\n" + "="*80)
    print("  Runtime Comparison")
    print("="*80)

    runtime_cols = ["training_time_s", "inference_time_s"]
    for col in runtime_cols:
        if col in df_all.columns:
            grouped = df_all.groupby("method")[col].agg(["mean", "std"]).round(2)
            print(f"\n{col}:")
            print(grouped.to_string())

## Part B: Ablation Study

In [ ]:
from src.utils.config import load_config, update_config
from src.training.trainer_3dgs import Trainer3DGS
from src.data.sparse_simulator import SparseSimulator

prepared_dir = OUTPUT_ROOT / "prepared"
with open(prepared_dir / "split_info.json") as f:
    split_info = json.load(f)

config = load_config("configs/default.yaml")

ablation_cases = split_info["test"][:ABLATION_CASES]
R = ABLATION_SPARSE_RATIO
organ_labels = {"liver": 1, "bladder": 2, "lungs": 3, "kidneys": 4, "bone": 5}

# Define ablation variants
ablation_variants = {
    "no_reg": {
        "loss": {"lambda_smooth": 0.0, "lambda_edge": 0.0},
        "desc": "No regularization",
    },
    "smooth_only": {
        "loss": {"lambda_smooth": 0.01, "lambda_edge": 0.0},
        "desc": "Smoothness only",
    },
    "edge_only": {
        "loss": {"lambda_smooth": 0.0, "lambda_edge": 0.005},
        "desc": "Edge preservation only",
    },
    "full": {
        "loss": {"lambda_smooth": 0.01, "lambda_edge": 0.005},
        "desc": "Full model (Ours)",
    },
    "grid_init": {
        "gaussian": {"init_mode": "grid"},
        "loss": {"lambda_smooth": 0.01, "lambda_edge": 0.005},
        "desc": "Grid initialization",
    },
    "adaptive_init": {
        "gaussian": {"init_mode": "adaptive"},
        "loss": {"lambda_smooth": 0.01, "lambda_edge": 0.005},
        "desc": "Adaptive (edge-aware) initialization",
    },
    "no_lr_decay": {
        "gaussian": {"lr_position_final": 0.01},  # Same as initial = no decay
        "loss": {"lambda_smooth": 0.01, "lambda_edge": 0.005},
        "desc": "No position LR decay",
    },
}

print(f"Ablation cases: {ablation_cases}")
print(f"Sparse ratio: R={R}")
print(f"Iterations: {ABLATION_ITERATIONS}")
print(f"Variants: {list(ablation_variants.keys())}")

In [ ]:
from tqdm import tqdm

ablation_results = []

for variant_name, variant_cfg in ablation_variants.items():
    print(f"\n{'='*60}")
    print(f"  Ablation: {variant_name} - {variant_cfg['desc']}")
    print(f"{'='*60}")

    for case_idx in tqdm(ablation_cases, desc=variant_name):
        vol_path = prepared_dir / f"volume_case{case_idx}.npy"
        if not vol_path.exists():
            continue

        volume = np.load(vol_path)
        labels = None
        lab_path = prepared_dir / f"labels_case{case_idx}.npy"
        if lab_path.exists():
            labels = np.load(lab_path)

        sim = SparseSimulator(sparse_ratio=R)
        sparse = sim.simulate(volume)

        cfg = deepcopy(config)
        cfg["gaussian"]["num_iterations"] = ABLATION_ITERATIONS

        # Apply variant overrides
        for section, overrides in variant_cfg.items():
            if section == "desc":
                continue
            if section not in cfg:
                cfg[section] = {}
            cfg[section].update(overrides)

        set_seed(SEED)

        ckpt_dir = str(CHECKPOINT_ROOT / "ablation" / variant_name / f"case{case_idx}_R{R}")

        trainer = Trainer3DGS(
            volume=volume,
            observed_indices=sparse["observed_indices"],
            target_indices=sparse["target_indices"],
            config=cfg,
            labels=labels,
            device=device,
            checkpoint_dir=ckpt_dir,
        )

        trainer.train()
        ev = trainer.evaluate_on_targets(organ_labels=organ_labels)
        s = ev["summary"]

        print(f"  Case {case_idx}: PSNR={s['mean_psnr']:.2f} | SSIM={s['mean_ssim']:.4f}")

        ablation_results.append({
            "variant": variant_name,
            "description": variant_cfg["desc"],
            "case_idx": int(case_idx),
            "sparse_ratio": R,
            **s,
        })

        del trainer
        torch.cuda.empty_cache()
        gc.collect()

print(f"\nTotal ablation experiments: {len(ablation_results)}")

In [ ]:
df_abl = pd.DataFrame(ablation_results)

# ============= PAPER TABLE: Ablation Study =============
print("\n" + "="*80)
print("  PAPER TABLE: Ablation Study Results")
print("="*80)

# Main ablation: Regularization components
reg_variants = ["no_reg", "smooth_only", "edge_only", "full"]
df_reg = df_abl[df_abl["variant"].isin(reg_variants)]

if len(df_reg) > 0:
    print("\n--- Regularization Ablation ---")
    reg_summary = df_reg.groupby("variant").agg({
        "mean_psnr": ["mean", "std"],
        "mean_ssim": ["mean", "std"],
    }).round(4)
    print(reg_summary.to_string())

# Init ablation
init_variants = ["grid_init", "adaptive_init"]
df_init = df_abl[df_abl["variant"].isin(init_variants)]

if len(df_init) > 0:
    print("\n--- Initialization Ablation ---")
    init_summary = df_init.groupby("variant").agg({
        "mean_psnr": ["mean", "std"],
        "mean_ssim": ["mean", "std"],
    }).round(4)
    print(init_summary.to_string())

# LR decay ablation
lr_variants = ["full", "no_lr_decay"]
df_lr = df_abl[df_abl["variant"].isin(lr_variants)]

if len(df_lr) > 0:
    print("\n--- LR Decay Ablation ---")
    lr_summary = df_lr.groupby("variant").agg({
        "mean_psnr": ["mean", "std"],
        "mean_ssim": ["mean", "std"],
    }).round(4)
    print(lr_summary.to_string())

# Save
abl_dir = OUTPUT_ROOT / "ablation"
abl_dir.mkdir(parents=True, exist_ok=True)
df_abl.to_csv(abl_dir / "ablation_results.csv", index=False)
print("\nSaved ablation_results.csv")

In [ ]:
# Statistical significance test (paired t-test)
from scipy import stats

if len(df_reg) > 0:
    print("\n=== Statistical Significance Tests ===")
    full_psnr = df_reg[df_reg["variant"] == "full"]["mean_psnr"].values

    for variant in ["no_reg", "smooth_only", "edge_only"]:
        other_psnr = df_reg[df_reg["variant"] == variant]["mean_psnr"].values
        if len(full_psnr) == len(other_psnr) and len(full_psnr) > 1:
            t_stat, p_val = stats.ttest_rel(full_psnr, other_psnr)
            diff = full_psnr.mean() - other_psnr.mean()
            print(f"  Full vs {variant}: diff={diff:+.3f} dB, p={p_val:.4f} "
                  f"{'***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'}")